# M6 — TabNet Employee Resignation Model

**Owner:** Member 6  
**Primary selection metric:** development out-of-fold average precision (PR-AUC)  
**Formal holdout status:** prohibited in this notebook

This notebook runs the frozen TabNet experiment contract. Category vocabularies, numerical imputation, and numerical scaling are learned inside each outer-training fold. The outer validation fold is used only for out-of-fold prediction; TabNet early stopping uses a stratified inner split created by the adapter.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path("../..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.metrics import calculate_binary_metrics
from src.models.runner import run_5fold_cv
from src.models.tabnet_model import TabNetAdapter, load_tabnet_config

CONFIG = load_tabnet_config(PROJECT_ROOT / "configs" / "models" / "tabnet.yaml")
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "Extended_Employee_Performance_and_Productivity_Data.csv"
REGISTRY_PATH = PROJECT_ROOT / "data" / "processed" / "split_registry.csv"
OUTPUT_DIR = PROJECT_ROOT / "results" / "models" / "tabnet"
FEATURES = CONFIG["preprocessing"]["categorical_features"] + CONFIG["preprocessing"]["numerical_features"]
TARGET = "Resigned"
ID_COLUMN = "Employee_ID"

print(f"Project root: {PROJECT_ROOT}")
print(f"TabNet candidates: {[item['id'] for item in CONFIG['candidates']]}")

Project root: C:\Users\Kawya\Documents\AIML Project
TabNet candidates: ['T01', 'T02', 'T03', 'T04', 'T05', 'T06', 'T07', 'T08']


## 1. Load only the frozen development partition

The registry—not a new random split—defines the 80,000 development rows and five shared folds. This cell explicitly checks that no formal-holdout employee enters the modelling dataframe.

In [2]:
raw = pd.read_csv(RAW_PATH)
registry = pd.read_csv(REGISTRY_PATH)

development_registry = registry.loc[registry["partition"] == "development", [ID_COLUMN, "cv_fold"]].copy()
holdout_ids = set(registry.loc[registry["partition"] == "holdout", ID_COLUMN])
development = raw.merge(development_registry, on=ID_COLUMN, how="inner", validate="one_to_one")

print(f"Raw shape: {raw.shape}")
print(f"Development shape after registry filtering: {development.shape}")
assert len(development) == 80_000
assert set(development[ID_COLUMN]).isdisjoint(holdout_ids)
assert sorted(development["cv_fold"].unique().tolist()) == [1, 2, 3, 4, 5]
assert set(FEATURES).issubset(development.columns)
assert ID_COLUMN not in FEATURES and "Hire_Date" not in FEATURES and TARGET not in FEATURES

target_summary = development[TARGET].value_counts(dropna=False).rename_axis(TARGET).to_frame("count")
target_summary["proportion"] = target_summary["count"] / len(development)
display(target_summary)

Raw shape: (100000, 20)
Development shape after registry filtering: (80000, 21)


,count,proportion
Resigned,,
False,71992,0.8999
True,8008,0.1001


## 2. Dependency and save/reload smoke test

Run this only after the pinned PyTorch and `pytorch-tabnet` packages are installed. It uses 5,000 rows from the training side of shared fold 1 and a small untouched subset of fold 1 for pipeline verification. These smoke metrics are diagnostic only and must not be reported as formal model results. The save/reload assertion verifies that serialized preprocessing and model parameters reproduce probabilities.

In [3]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    smoke_config = CONFIG["smoke_test"]
    outer_training = development.loc[development["cv_fold"] != 1].reset_index(drop=True)
    outer_validation = development.loc[development["cv_fold"] == 1].reset_index(drop=True)

    smoke_train_idx, _ = train_test_split(
        np.arange(len(outer_training)),
        train_size=int(smoke_config["sample_rows"]),
        random_state=int(CONFIG["training"]["random_seed"]),
        stratify=outer_training[TARGET],
    )
    smoke_valid_idx, _ = train_test_split(
        np.arange(len(outer_validation)),
        train_size=1_000,
        random_state=int(CONFIG["training"]["random_seed"]),
        stratify=outer_validation[TARGET],
    )
    smoke_train = outer_training.iloc[smoke_train_idx]
    smoke_valid = outer_validation.iloc[smoke_valid_idx]
    print(f"Smoke training shape before/after sampling: {outer_training.shape} -> {smoke_train.shape}")
    print(f"Smoke validation shape before/after sampling: {outer_validation.shape} -> {smoke_valid.shape}")

    smoke_adapter = TabNetAdapter(
        config=CONFIG,
        candidate_id=smoke_config["candidate_id"],
        max_epochs_override=int(smoke_config["max_epochs"]),
        patience_override=int(smoke_config["patience"]),
    )
    smoke_adapter.fit(smoke_train[FEATURES], smoke_train[TARGET])
    smoke_probability = smoke_adapter.predict_proba(smoke_valid[FEATURES])
    smoke_metrics = calculate_binary_metrics(smoke_valid[TARGET].astype(int), smoke_probability)
    display(pd.Series(smoke_metrics, name="diagnostic_smoke_metric"))

    smoke_artifact_dir = OUTPUT_DIR / "smoke_artifact"
    smoke_adapter.save(smoke_artifact_dir)
    reloaded_adapter = TabNetAdapter.load(smoke_artifact_dir)
    reloaded_probability = reloaded_adapter.predict_proba(smoke_valid[FEATURES])
    np.testing.assert_allclose(smoke_probability, reloaded_probability, rtol=1e-6, atol=1e-7)
    print("Smoke fit, prediction, serialization, and reload checks passed.")
else:
    print("Smoke training is disabled. Set RUN_SMOKE_TEST=True only after dependency approval.")

Smoke training is disabled. Set RUN_SMOKE_TEST=True only after dependency approval.


## 3. Formal candidate run on the five shared development folds

Run one declared candidate at a time and retain every OOF result. Candidate selection happens only after comparing all completed development OOF results by pooled average precision, with runtime and stability as secondary evidence. Do not open, score, or predict the formal holdout here.

In [4]:
RUN_FORMAL_CV = False
CANDIDATE_TO_RUN = "T05"

if RUN_FORMAL_CV:
    declared_ids = {item["id"] for item in CONFIG["candidates"]}
    assert CANDIDATE_TO_RUN in declared_ids
    result = run_5fold_cv(
        adapter_factory=lambda: TabNetAdapter(config=CONFIG, candidate_id=CANDIDATE_TO_RUN),
        data=development.drop(columns=["cv_fold"]),
        split_registry=registry,
        feature_cols=FEATURES,
        target_col=TARGET,
        expected_model_name="TabNet",
    )
    candidate_dir = OUTPUT_DIR / CANDIDATE_TO_RUN.lower()
    candidate_dir.mkdir(parents=True, exist_ok=True)
    result.predictions.to_csv(candidate_dir / "oof_predictions.csv", index=False)
    result.fold_metrics.to_csv(candidate_dir / "fold_metrics.csv", index=False)
    (candidate_dir / "summary_metrics.json").write_text(
        json.dumps(result.summary_metrics, indent=2), encoding="utf-8"
    )
    display(result.fold_metrics)
    display(pd.Series(result.summary_metrics, name=CANDIDATE_TO_RUN))
else:
    print("Formal five-fold TabNet training is disabled until the smoke and reload checks pass.")

C:\Users\Kawya\Documents\AIML Project\.venv\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.41084 | inner_valid_average_precision: 0.10255 |  0:00:02s


epoch 1  | loss: 0.33333 | inner_valid_average_precision: 0.10407 |  0:00:04s


epoch 2  | loss: 0.33111 | inner_valid_average_precision: 0.10683 |  0:00:06s


epoch 3  | loss: 0.32873 | inner_valid_average_precision: 0.0987  |  0:00:08s


epoch 4  | loss: 0.32896 | inner_valid_average_precision: 0.1058  |  0:00:10s


... (118 outputs left out)

epoch 7  | loss: 0.32741 |  0:00:16s


epoch 8  | loss: 0.32757 |  0:00:18s


,average_precision,roc_auc,precision,recall,f1,balanced_accuracy,accuracy,brier_score,tn,fp,fn,tp,predicted_positive_count,predicted_positive_rate,fold,fit_time_seconds,predict_time_seconds
0,0.098938,0.494347,0.0,0.0,0.0,0.5,0.899937,0.091269,14399,0,1601,0,0,0.0,1,47.770138,0.253809
1,0.101636,0.503355,0.0,0.0,0.0,0.5,0.899937,0.091073,14399,0,1601,0,0,0.0,2,40.088360,0.228045
2,0.100962,0.502601,0.0,0.0,0.0,0.5,0.899875,0.091600,14398,0,1602,0,0,0.0,3,44.739610,0.343722
3,0.101833,0.502776,0.0,0.0,0.0,0.5,0.899875,0.091820,14398,0,1602,0,0,0.0,4,36.156058,0.227719
4,0.101531,0.501146,0.0,0.0,0.0,0.5,0.899875,0.091387,14398,0,1602,0,0,0.0,5,65.550352,0.261444


model_name                        TabNet
model_id                          tabnet
total_development_records          80000
total_folds                            5
pooled_average_precision        0.100672
pooled_roc_auc                  0.501184
pooled_f1                            0.0
pooled_brier_score               0.09143
cv_mean_average_precision        0.10098
cv_std_average_precision        0.001187
cv_mean_roc_auc                 0.500845
cv_std_roc_auc                  0.003723
cv_mean_f1                           0.0
cv_std_f1                            0.0
total_fit_time_seconds        234.304518
total_predict_time_seconds       1.31474
Name: T05, dtype: object

## Git checkpoint

Commit the TabNet configuration, adapter, tests, and this clean unexecuted scaffold after the core tests pass. Push the trained-result commit only after the dependency smoke test, save/reload check, five-fold OOF validation, and notebook Restart & Run All all pass. Never commit the raw employee dataset, local virtual environment, notebook checkpoints, or temporary smoke artifacts.